# Fixed data center: native EGO HPP sizing
Use **hydesign_clean** and Run All. The shared YAML supplies defaults; edit the single configuration cell below.
The original Parallel EGO runs in a separate process with a wall-clock limit. Completed evaluations are saved,
and the cheapest feasible candidate is independently checked after the search stops. This final check is outside
the optimization budget. Results are feasible incumbents, not certified global optima.

Shared infrastructure is always included: G_proxy_MW = facility_load_mw + rated b_P,
without efficiency correction. Native shared_cost uses BOS 119,940 EUR/MW and
land 300,000 EUR/km2 times max(Awpp, Apvp); public-grid cost is zero.
This capacity proxy is not a detailed electrical-network design. Shared CAPEX
is annualized using the HPP rates (6%) and economic lifetime (25 years).

Battery economics include one full replacement in year 15, discounted at the
battery WACC (6%) and annualized across the HPP economic lifetime (25 years).
CAPEX_b and CAPEX_hpp remain year-0 investment only. The new replacement fields
are expenditures in EUR, not EUR/year. There is no physical degradation, price
reduction or salvage value; hourly dispatch and reliability are unchanged.


In [ ]:
import os
os.environ['OPENMDAO_REPORTS'] = '0'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hydesign
from hydesign.examples import examples_filepath
from hydesign.sizing_native_hpp_sizing_base import read_configuration, run_native_hpp_sizing
from hydesign.assembly.hpp_assembly_simple_datacenter_hpp_sizing_base import VARIABLE_NAMES

repo = Path(hydesign.__file__).resolve().parent.parent
print('Python:', sys.executable)
print('Repository:', repo)
config_path = Path(examples_filepath) / 'thesis_simple_datacenter/simple_datacenter.yml'


## Configuration
All changes below apply to this run only; the shared YAML is unchanged. Switch each capacity between
`fixed` and `design`. Fixed variables use `value`; optimized variables use `limits` and their native type.
All fixed evaluates once. Set the outer time limit to `None` to disable it.


In [ ]:
# Defaults are read from YAML. Uncomment a line to override this run only.
config = read_configuration(config_path)

# Data center and reliability
config['facility_load_mw'] = 16
config['reliability_target'] = 0.99
# config['pue'] = 1.15

# Optimization runtime and output
config['hpp_sizing_time_limit_seconds'] = 1800  # None disables the outer limit.
config['hpp_sizing_output_dir'] = str(repo / 'hpp_sizing_runs')
# config['hpp_sizing_site'] = 'Denmark_good_solar'

# Design variables: keep geometry and turbine rating fixed.
variables = config['variables']
variables['Nwt']['var_type'] = 'design'
variables['solar_MW [MW AC]']['var_type'] = 'design'
variables['b_P [MW]']['var_type'] = 'design'
variables['b_E_h [h]']['var_type'] = 'design'

# Optional fixed values and design bounds
# variables['Nwt']['value'] = 20
# variables['solar_MW [MW AC]']['value'] = 180
# variables['b_P [MW]']['value'] = 25
# variables['b_E_h [h]']['value'] = 8
# variables['Nwt']['limits'] = [0, 400]

# Original Parallel EGO settings (otherwise use YAML)
# config['n_doe'] = 16
# config['n_clusters'] = 4
# config['n_procs'] = 2
# config['max_iter'] = 20
# config['npred'] = 2000
config['n_seed'] = 1
# config['tol'] = 1e-6
# config['min_conv_iter'] = 3

# Inner EMS solver and battery dispatch
# config['ems_time_limit_seconds'] = 600
# config['ems_threads'] = 1
# config['battery_power_fluctuation_penalty'] = 0.0001

print('Outer budget / inner solve limit [s]:',
      config['hpp_sizing_time_limit_seconds'], config['ems_time_limit_seconds'])
print('Facility / installed IT [MW]:', config['facility_load_mw'], config['facility_load_mw'] / config['pue'])
print('Reliability / allowed failed hours:', config['reliability_target'],
      int(np.floor((1-config['reliability_target'])*8760 + 1e-9)))
display(pd.DataFrame(variables).T)


## Model setup and optimization
The supporting module resolves the site and physical model, then launches the unchanged native driver.
The limit covers startup, DOE, dispatch and surrogate iterations. Evaluation records and native progress
are saved in the run directory. No previous custom search implementation is used.


In [ ]:
result = run_native_hpp_sizing(config_path, overrides=config)
print('Results:', result['run_directory'])


## Results
`TIME_LIMIT` means the best recovered feasible incumbent is reported. `NATIVE_CONVERGENCE` and
`MAX_ITERATIONS` distinguish the original driver's normal stopping conditions. Interrupted evaluations
remain unresolved. Final reliability, SOC, power balance and simultaneous-operation checks must pass.


In [ ]:
print('Termination:', result['termination_reason'])
print('Configured limit / optimization / cleanup [s]:',
      result['configuration']['hpp_sizing_time_limit_seconds'],
      result['optimization_seconds'], result['cleanup_seconds'])
print('Independent final validation [s]:', result['final_validation_seconds'])
print('Completed / attempted / unique designs:',
      result['evaluated_designs'], result['attempted_evaluations'], result['unique_designs'])
print('Feasible / infeasible / unresolved evaluations:',
      result['feasible_evaluations'], result['infeasible_evaluations'], result['unresolved_evaluations'])
print('Process cleanup:', result['process_cleanup'])
print('Final status:', result['final_status'])
if result['error'] or result['validation_errors']:
    print('Errors:', result['error'], result['validation_errors'])
best = result['best']
if best is None:
    print('No independently validated feasible design found; this does not prove the domain infeasible.')
else:
    display(pd.Series(dict(zip(VARIABLE_NAMES, best['design'])), name='Best validated HPP design'))
    print('Wind capacity [MW]:', best['design'][3] * best['design'][2])
    print('Battery power [MW] / duration [h] / energy [MWh]:',
          best['design'][-2], best['design'][-1], best['design'][-2] * best['design'][-1])
    print('Fixed DC facility load [MW]:', result['configuration']['facility_load_mw'])
    print('Target / achieved reliability [%]:',
          100 * result['configuration']['reliability_target'], 100 * best['metrics']['firm_reliability'])
    print('Failed hours / EENS [MWh]:', best['metrics']['LOLH'], best['metrics']['EENS'])
    print('HPP CAPEX [EUR] / annualized cost [EUR/year]:',
          best['economics']['CAPEX_hpp'], best['economics']['annualized_cost_hpp'])
    print('Solver status:', best['solver_status']['stages'][-1]['status'])
    print('Time-limited incumbent:', result['time_limited_incumbent'])
    print('Shared proxy [MW] / BOS CAPEX / land CAPEX / shared CAPEX [EUR] / annual cost [EUR/year]:')
    display(pd.Series({k: best['economics'][k] for k in ('G_proxy_MW', 'CAPEX_sh_fixed', 'CAPEX_sh_land', 'CAPEX_sh', 'annualized_cost_shared', 'net_annual_value_system')}))
    print('Battery: initial / replacement / discounted replacement [EUR]; annualized CAPEX / cost [EUR/year]:')
    display(pd.Series({k: best['economics'][k] for k in ('CAPEX_b', 'battery_replacement_cost', 'discounted_battery_replacement_cost', 'annualized_CAPEX_battery', 'annualized_cost_battery')}))


In [ ]:
history = pd.read_csv(result['history_path'])
if not history.empty:
    history.plot(x='elapsed_seconds', y='best_feasible_cost', marker='o', legend=False)
    plt.ylabel('Best feasible annual HPP cost [EUR/year]')
    plt.xlabel('Optimization elapsed time [s]')
    plt.show()
else:
    print('No evaluation started before the search stopped.')


Reliability stays a hard hourly EMS constraint; EENS and DC economics are reporting quantities.
The history plot uses completed physical evaluations, so it also works when EGO stops before publishing
its recorder. Native EGO retains its existing surrogate and constant rejection scores; convergence can
still be difficult when many designs are infeasible. Increasing the outer time limit does not change the
per-dispatch CPLEX limit. Final validation can take additional time; its duration is reported separately.
